# Math Engine - Interactive Demo

This notebook walks through the core quantitative primitives used in the Sports Betting Arbitrage Scanner.

**Modules covered:**
1. Odds Conversion (American -> Decimal -> Implied Probability)
2. Vig Calculation & De-Vigging (Fair Value from Pinnacle)
3. Expected Value (+EV%) Calculation
4. Kelly Criterion Bet Sizing

In [ ]:
import sys
sys.path.insert(0, '..')

from src.math_engine import MathEngine

m = MathEngine()
print('MathEngine loaded successfully')

## 1. Odds Conversion

American odds are the standard in US sportsbooks:
- **Positive (+150):** How much you win on a $100 bet
- **Negative (-200):** How much you must bet to win $100

Decimal odds are simpler: `payout = stake * decimal_odds`

In [ ]:
# Underdog: +150 American
dec = m.american_to_decimal(+150)
prob = m.decimal_to_implied_probability(dec)
print(f'+150 American = {dec} Decimal = {prob:.1%} Implied Probability')
print(f'  -> $100 bet pays ${100 * dec:.0f} total (${100 * (dec-1):.0f} profit)\n')

# Favorite: -200 American
dec2 = m.american_to_decimal(-200)
prob2 = m.decimal_to_implied_probability(dec2)
print(f'-200 American = {dec2} Decimal = {prob2:.1%} Implied Probability')
print(f'  -> $200 bet pays ${200 * dec2:.0f} total (${200 * (dec2-1):.0f} profit)')

## 2. Vig Removal (De-Vigging)

Bookmakers embed a margin ("vig" / "juice") into their odds. A standard -110/-110 line has ~4.76% vig.

**Why this matters:** We strip the vig from Pinnacle (the sharpest book) to get **true probabilities** - our source of truth for finding +EV bets elsewhere.

In [ ]:
# Simulate a Pinnacle line: Lakers -160 / Celtics +140
lakers_dec = m.american_to_decimal(-160)
celtics_dec = m.american_to_decimal(+140)

lakers_imp = m.decimal_to_implied_probability(lakers_dec)
celtics_imp = m.decimal_to_implied_probability(celtics_dec)

print(f'Pinnacle Raw Odds:')
print(f'  Lakers:  -160 = {lakers_dec} dec = {lakers_imp:.4f} implied')
print(f'  Celtics: +140 = {celtics_dec} dec = {celtics_imp:.4f} implied')
print(f'  Sum: {lakers_imp + celtics_imp:.4f} (should be > 1.0)\n')

# Calculate and remove vig
vig = m.calculate_vig(lakers_imp, celtics_imp)
print(f'Vig: {vig:.4f} ({vig*100:.2f}%)\n')

true_lakers, true_celtics = m.devig_probabilities(lakers_imp, celtics_imp)
print(f'TRUE (Fair) Probabilities:')
print(f'  Lakers:  {true_lakers:.4f} ({true_lakers*100:.1f}%)')
print(f'  Celtics: {true_celtics:.4f} ({true_celtics*100:.1f}%)')
print(f'  Sum: {true_lakers + true_celtics:.6f} (exactly 1.0)\n')

fair_lakers = m.true_probability_to_fair_odds(true_lakers)
fair_celtics = m.true_probability_to_fair_odds(true_celtics)
print(f'Fair Decimal Odds (no vig):')
print(f'  Lakers:  {fair_lakers}')
print(f'  Celtics: {fair_celtics}')

## 3. Expected Value (+EV%)

The core alpha signal. If DraftKings is offering Celtics at +155 but Pinnacle's true probability gives fair odds of +145, DraftKings is overpaying.

**Formula:** `EV% = (true_prob * decimal_offered) - 1`

In [ ]:
# DraftKings offers Celtics at +155 (decimal 2.55)
dk_celtics_dec = m.american_to_decimal(+155)

# True probability from Pinnacle de-vig (calculated above)
ev = m.expected_value(true_celtics, dk_celtics_dec)

print(f'DraftKings Celtics: +155 = {dk_celtics_dec} dec')
print(f'Pinnacle True Prob: {true_celtics:.4f}')
print(f'Fair Odds:          {fair_celtics}')
print(f'\nEV = ({true_celtics:.4f} x {dk_celtics_dec}) - 1 = {ev:.4f}')
print(f'\n>>> This bet has {ev*100:+.2f}% EV {"-- TAKE IT" if ev > 0.015 else "-- skip"}')

## 4. Kelly Criterion - Bet Sizing

Kelly tells us the **optimal fraction of bankroll** to risk given our edge and the odds.

**Formula:** `f* = (p*b - q) / b`

In practice, we use **fractional Kelly** (Quarter Kelly = 0.25) to reduce variance.

In [ ]:
bankroll = 1000  # $1,000

print(f'Bankroll: ${bankroll:,.0f}')
print(f'True Win Prob: {true_celtics:.4f}')
print(f'DK Odds: {dk_celtics_dec}\n')

for label, mult in [('Full Kelly', 1.0), ('Half Kelly', 0.5), ('Quarter Kelly', 0.25)]:
    frac = m.kelly_criterion(true_celtics, dk_celtics_dec, kelly_multiplier=mult)
    bet = m.kelly_bet_size(bankroll, true_celtics, dk_celtics_dec, kelly_multiplier=mult)
    print(f'{label:15s}: {frac:.4f} ({frac*100:.2f}%) = ${bet:.2f}')

## Edge Case Handling

The engine validates all inputs and raises clear errors:

In [ ]:
# Test edge cases
test_cases = [
    ('Odds = 0', lambda: m.american_to_decimal(0)),
    ('Decimal <= 1.0', lambda: m.decimal_to_implied_probability(0.5)),
    ('Probability = 0', lambda: m.devig_probabilities(0, 0.5)),
    ('Probability > 1', lambda: m.implied_probability_to_decimal(1.5)),
]

for name, fn in test_cases:
    try:
        fn()
        print(f'  {name}: NO ERROR (unexpected)')
    except ValueError as e:
        print(f'  {name}: ValueError raised -- {e}')

# Negative edge returns 0 bet
no_edge = m.kelly_criterion(0.30, 2.10)
print(f'\n  Negative edge Kelly: {no_edge} (correctly returns 0 = no bet)')